In [0]:
from pyspark.sql.functions import *

In [0]:
# Gold: patient_360 — one row per patient (STTM: GOLD/sttm_silver_to_gold.xlsx, sheet 'patient_360')
# Load type: FULL rebuild (CREATE OR REPLACE TABLE ... AS SELECT). LEFT JOINs from patients so no rows are dropped.
catalog = "caresyncsydatabricks"

spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.patient_360 AS
WITH lab_agg AS (
    SELECT
        patient_id,
        COUNT(*)                                                AS total_tests,
        COUNT_IF(result_status = 'Abnormal')                    AS abnormal_test_count,
        COUNT_IF(result_status = 'Critical')                    AS critical_test_count,
        MAX(test_date)                                          AS last_test_date
    FROM {catalog}.silver.lab_results
    GROUP BY patient_id
)
SELECT
    CAST(p.patient_id AS INT)                                                   AS patient_id,
    p.first_name,
    p.last_name,
    CAST(p.date_of_birth AS DATE)                                               AS date_of_birth,
    CAST(FLOOR(DATEDIFF(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
    p.gender,
    p.city,
    p.state,
    CAST(p.registered_hospital_id AS INT)                                       AS hospital_id,
    h.hospital_name,
    CAST(p.insurance_provider_id AS INT)                                        AS insurance_provider_id,
    ip.provider_name                                                            AS insurance_provider_name,
    CAST(COALESCE(l.total_tests, 0) AS INT)                                     AS total_tests,
    CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                             AS abnormal_test_count,
    CAST(COALESCE(l.critical_test_count, 0) AS INT)                             AS critical_test_count,
    l.last_test_date
FROM {catalog}.silver.patients p
LEFT JOIN {catalog}.silver.hospitals h
       ON p.registered_hospital_id = h.hospital_id
LEFT JOIN {catalog}.silver.insurance_providers ip
       ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg l
       ON p.patient_id = l.patient_id
""")

gold_patient_360 = spark.table(f"{catalog}.gold.patient_360")
display(gold_patient_360.limit(10))

In [0]:
# Validation: grain (one row per patient) and row count parity with silver.patients
display(spark.sql(f"""
SELECT
    (SELECT COUNT(*) FROM {catalog}.silver.patients)            AS silver_patients,
    COUNT(*)                                                    AS gold_rows,
    COUNT(DISTINCT patient_id)                                  AS distinct_patients,
    SUM(total_tests)                                            AS total_tests,
    (SELECT COUNT(*) FROM {catalog}.silver.lab_results)         AS silver_lab_results,
    COUNT_IF(hospital_name IS NULL)                             AS missing_hospital_name,
    COUNT_IF(insurance_provider_name IS NULL)                   AS missing_insurance_name,
    COUNT_IF(total_tests = 0)                                   AS patients_without_tests
FROM {catalog}.gold.patient_360
"""))

In [0]:
record_count = (spark.read.table("caresyncsydatabricks.gold.patient_360").count())

dbutils.notebook.exit(str(record_count))